In [21]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)

In [22]:
eval_iters = 200
batch_size = 32
eval_interval = 300
block_size = 8
max_iters = 5000
learning_rate = 1e-2

In [23]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print ("device: ", device)

device:  cpu


In [24]:
with open ("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

In [25]:
print("length of dataset in chars: ", len(text))

length of dataset in chars:  1115394


In [26]:
print(text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [27]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print("characters:", "".join(chars))
print("vocabulary size: ", vocab_size)

characters: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
vocabulary size:  65


In [28]:
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: "".join (itos[i] for i in l)

In [29]:
print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [30]:
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.type)
print(data[:100])

torch.Size([1115394]) <built-in method type of Tensor object at 0x1268a76b0>
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [31]:
n = int(0.9 * len (data))

train_data = data[:n]
val_data = data[n:]

In [32]:
def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

xb, yb =  get_batch("train")

print ("inputs")
print(xb.shape)
print(xb)

print ("targets")
print(yb.shape)
print(yb)

print ("-----------")

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b, t]
        print (f"when the input is {context}, the target is {target}")


inputs
torch.Size([32, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54],
        [57, 43, 60, 43, 52,  1, 63, 43],
        [60, 43, 42,  8,  0, 25, 63,  1],
        [56, 42,  5, 57,  1, 57, 39, 49],
        [43, 57, 58, 63,  6,  1, 58, 46],
        [43,  1, 51, 39, 63,  1, 40, 43],
        [58, 46, 43,  1, 43, 39, 56, 57],
        [39, 58, 47, 53, 52, 12,  1, 37],
        [53, 56, 43,  1, 21,  1, 41, 39],
        [50, 39, 52, 63,  1, 47, 58, 57],
        [56, 53, 63,  1, 42, 47, 42,  1],
        [39, 51,  1, 39, 44, 56, 39, 47],
        [17, 24, 21, 38, 13, 14, 17, 32],
        [ 1, 39, 52, 42,  1, 45, 43, 50],
        [ 1, 58, 46, 39, 58,  1, 42, 53],
        [ 1, 61, 53, 59, 50, 42,  1, 21],
        [59, 57, 40, 39, 52, 42,  1, 40],
        [52, 42,  8,  0,  0, 23, 21, 26],
        [45, 53, 42, 57,  0, 23, 43, 43],
        [52,  1, 61, 39, 57,  1, 51, 53],
       

when the input is tensor([24, 43, 58,  5, 57,  1, 46]), the target is 43
when the input is tensor([24, 43, 58,  5, 57,  1, 46, 43]), the target is 39
when the input is tensor([44]), the target is 53
when the input is tensor([44, 53]), the target is 56
when the input is tensor([44, 53, 56]), the target is 1
when the input is tensor([44, 53, 56,  1]), the target is 58
when the input is tensor([44, 53, 56,  1, 58]), the target is 46
when the input is tensor([44, 53, 56,  1, 58, 46]), the target is 39
when the input is tensor([44, 53, 56,  1, 58, 46, 39]), the target is 58
when the input is tensor([44, 53, 56,  1, 58, 46, 39, 58]), the target is 1
when the input is tensor([52]), the target is 58
when the input is tensor([52, 58]), the target is 1
when the input is tensor([52, 58,  1]), the target is 58
when the input is tensor([52, 58,  1, 58]), the target is 46
when the input is tensor([52, 58,  1, 58, 46]), the target is 39
when the input is tensor([52, 58,  1, 58, 46, 39]), the target i

In [33]:
class BigramLanguageModel(nn.Module):

    def __init__ (self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward (self, idx, targets = None):
        logits = self.token_embedding_table(idx)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate (self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx)
            logits = logits [:, -1, :]
            probs = F.softmax (logits, dim = -1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim = 1)
        return idx

model = BigramLanguageModel(vocab_size)
model = model.to(device)

In [34]:
total_params = sum(p.numel() for p in model.parameters())
print("model parameters count: ", total_params)

model parameters count:  4225


In [35]:
logits, loss = model(xb, yb)

print(logits.shape)
print(loss)

torch.Size([256, 65])
tensor(4.6922, grad_fn=<NllLossBackward0>)


In [36]:
idx = torch.zeros((1, 1), dtype=torch.long)
print (decode(model.generate(idx, max_new_tokens=100)[0].tolist()))


JLg,3D&OM .3YCjfolRwqXaDyttW!GmaUT-IIvuZV?sYfjzUvTQ3RwL ?etyLeg.COHW
Ri$ELkJMXpBEX;-G&Orl!bcH ;cq.z,


In [37]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

In [38]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range (eval_iters):
            X, Y = get_batch(split)
            _, loss = model (X, Y)
            losses [k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out        

In [39]:
for iter in range(max_iters):

    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses["train"]:.4f}, val loss = {losses["val"]:.4f}")

    xb, yb = get_batch("train")
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()


step 0: train loss 4.7657, val loss = 4.7640
step 300: train loss 2.8276, val loss = 2.8493
step 600: train loss 2.5497, val loss = 2.5785
step 900: train loss 2.4998, val loss = 2.5239
step 1200: train loss 2.4756, val loss = 2.5033
step 1500: train loss 2.4670, val loss = 2.5002
step 1800: train loss 2.4696, val loss = 2.5010
step 2100: train loss 2.4668, val loss = 2.4888
step 2400: train loss 2.4580, val loss = 2.4928
step 2700: train loss 2.4672, val loss = 2.4983
step 3000: train loss 2.4592, val loss = 2.4882
step 3300: train loss 2.4701, val loss = 2.4920
step 3600: train loss 2.4599, val loss = 2.4868
step 3900: train loss 2.4623, val loss = 2.4896
step 4200: train loss 2.4582, val loss = 2.4932
step 4500: train loss 2.4722, val loss = 2.4884
step 4800: train loss 2.4567, val loss = 2.4926


In [40]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print (decode(model.generate(context, max_new_tokens=500)[0].tolist()))


Lar y
Y:
cht port.
O t gn an bat,

Thelllaterfucover;

ththamend hamspel mesitithen witus ise ingouters, fe brnddlisar res!
Yok&My limangoued herds c. car.
Dowhave, tele womu winds, the ir He sexpomes therar hacy kevislthy de m, ce d d colinomo r m at os hofr. f--t; he woinoseshe fewin: mo owe
The apukerd mpheand ad s f retnt she hutos ouroug.
Onokeand ameco, w n tt hed, anon trarithothen
Wheok t br thakeldgou m se d ind f my YOfok tofor r, Pllanloroublled myouspibolllld'd linse cetorderu m ur i
